# Chicken-threat candidate training
Run this launcher in Kaggle with the approved dataset and fixed evaluation inputs attached.

In [ ]:
REPOSITORY_URL = "https://github.com/Dumdart/SmartHomeBridge.git"
COMMIT_SHA = "SET_THE_REVIEWED_COMMIT_SHA"
DATASET_YAML = "/kaggle/input/chicken-threat-dataset-v5/data.yaml"
V4_TEST_YAML = "/kaggle/input/chicken-threat-v4-test/data.yaml"
BARN_HOLDOUT_YAML = "/kaggle/input/chicken-threat-barn-holdout/data.yaml"
OUTPUT_DIR = "/kaggle/working/chicken-threat-candidate"
MODEL_ID = "chicken-threat-yolo11m-v5.0.0"
DATASET_MANIFEST = "/kaggle/input/chicken-threat-dataset-v5/dataset_manifest.json"

In [ ]:
!git clone $REPOSITORY_URL SmartHomeBridge
%cd SmartHomeBridge
!git checkout $COMMIT_SHA
%pip install -q uv
!uv sync --extra ml --locked
!uv run smart-home-ml-train --dataset-yaml $DATASET_YAML --training-config ml/chicken_threat/configs/training.yaml --output-dir $OUTPUT_DIR

In [ ]:
BEST_WEIGHTS = f"{OUTPUT_DIR}/chicken-threat-yolo11m-v5/weights/best.pt"
!uv run smart-home-ml-evaluate --weights $BEST_WEIGHTS --dataset-yaml $V4_TEST_YAML --class-mapping ml/chicken_threat/configs/class_mapping.yaml --output $OUTPUT_DIR/v4_test.json --evaluation-name v4_test
!uv run smart-home-ml-evaluate --weights $BEST_WEIGHTS --dataset-yaml $BARN_HOLDOUT_YAML --class-mapping ml/chicken_threat/configs/class_mapping.yaml --output $OUTPUT_DIR/barn_holdout.json --evaluation-name barn_holdout
!uv run smart-home-ml-package-candidate --weights $BEST_WEIGHTS --dataset-manifest $DATASET_MANIFEST --class-mapping ml/chicken_threat/configs/class_mapping.yaml --evaluation $OUTPUT_DIR/v4_test.json --evaluation $OUTPUT_DIR/barn_holdout.json --training-config ml/chicken_threat/configs/training.yaml --output-dir $OUTPUT_DIR --model-id $MODEL_ID

Evaluate the resulting `best.pt` on both supplied fixed inputs with `smart-home-ml-evaluate`, then package the candidate. Candidate selection and promotion remain manual.